In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import plotly.express as px


hno_df = pd.read_csv('../../data/hpc_hno_2024.csv', low_memory=False)
fts_req_df = pd.read_csv('../../data/fts_requirements_funding_global.csv')
hno_df = hno_df[1:]
fts_req_df = fts_req_df[1:]

In [16]:
fts_2024 = fts_req_df[fts_req_df['year'] == 2024].copy()
fts_2024
# fts_2024['requirements'] = pd.to_numeric(fts_2024['requirements'], errors='coerce')
# fts_2024['funding'] = pd.to_numeric(fts_2024['funding'], errors='coerce')
# fts_2024['percentFunded'] = pd.to_numeric(fts_2024['percentFunded'], errors='coerce')



,countryCode,id,name,code,typeId,typeName,startDate,endDate,year,requirements,funding,percentFunded
7,AFG,1185.0,Afghanistan Humanitarian Needs and Response Pl...,HAFG24,2070.0,Humanitarian needs and response plan,2024-01-01,2024-12-31,2024,3.059588e+09,1.583614e+09,52.0
8,AFG,NaN,Not specified,NaN,NaN,NaN,NaN,NaN,2024,NaN,1.154539e+08,NaN
62,AGO,1164.0,Democratic Republic of the Congo Regional Refu...,RDRC_RRP24,111.0,Regional response plan,2024-01-01,2024-12-31,2024,2.064627e+07,9.561500e+04,0.0
63,AGO,NaN,Not specified,NaN,NaN,NaN,NaN,NaN,2024,NaN,1.000812e+07,NaN
107,ARG,1163.0,Venezuela Regional Refugee and Migrant Respons...,RREG24,111.0,Regional response plan,2024-01-01,2024-12-31,2024,2.202653e+07,1.251020e+06,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...
3724,ZMB,1198.0,Zambia Drought Flash Appeal 2024,FZMB24,5.0,Flash appeal,2024-05-01,2024-12-31,2024,1.060722e+08,5.805845e+07,55.0
3725,ZMB,1164.0,Democratic Republic of the Congo Regional Refu...,RDRC_RRP24,111.0,Regional response plan,2024-01-01,2024-12-31,2024,3.217212e+07,6.338400e+05,2.0
3726,ZMB,NaN,Not specified,NaN,NaN,NaN,NaN,NaN,2024,NaN,3.117186e+06,NaN
3764,ZWE,1199.0,Zimbabwe Drought Flash Appeal 2024,FZWE24,5.0,Flash appeal,2024-05-01,2024-12-31,2024,2.861911e+08,1.046380e+08,37.0


In [ ]:
hno_df['In Need'] = pd.to_numeric(hno_df['In Need'], errors='coerce')
hno_df['Targeted'] = pd.to_numeric(hno_df['Targeted'], errors='coerce')
hno_df['Population'] = pd.to_numeric(hno_df['Population'], errors='coerce')

hno_country = hno_df.groupby('Country ISO3').agg({
    'In Need': 'sum',
    'Targeted': 'sum',
    'Population': 'max'
}).reset_index()

df = pd.merge(
    hno_country, 
    fts_2024, 
    left_on='Country ISO3', 
    right_on='countryCode', 
    how='inner'
)

df['Need_Density'] = df['In Need'] / df['Population']
df['Targeting_Coverage'] = df['Targeted'] / df['In Need']
df['Log_Requirements'] = np.log10(df['requirements'] + 1)

scaler = MinMaxScaler()
severity_features = ['Need_Density', 'Log_Requirements']
df[['Norm_Density', 'Norm_Req']] = scaler.fit_transform(df[severity_features])

df['Severity_Score'] = (df['Norm_Density'] + df['Norm_Req']) / 2
df['Norm_Funding'] = df['percentFunded'] / 100
df['Mismatch_Score'] = df['Severity_Score'] - df['Norm_Funding']

df = df.sort_values(by='Mismatch_Score', ascending=False)

top_overlooked = df[['name', 'Mismatch_Score', 'Severity_Score', 'percentFunded', 'In Need']].head(10)
print("Top 10 Most Overlooked Crises:")
print(top_overlooked)

fig = px.scatter(
    df, 
    x='percentFunded', 
    y='Severity_Score', 
    text='name',
    size='In Need',
    color='Mismatch_Score',
    color_continuous_scale='RdYlGn_r', # Red is high mismatch, Green is well-funded
    title='Crisis Mismatch Analysis: Funding vs. Severity',
    labels={'percentFunded': 'Funding Coverage (%)', 'Severity_Score': 'Composite Severity Score (0-1)'}
)
fig.add_vline(x=df['percentFunded'].mean(), line_dash="dash", annotation_text="Avg Funding")
fig.add_hline(y=df['Severity_Score'].mean(), line_dash="dash", annotation_text="Avg Severity")
fig.update_traces(textposition='top center')
fig.show()

Top 10 Most Overlooked Crises:
                                                 name  Mismatch_Score  \
0   Afghanistan Humanitarian Needs and Response Pl...        0.456031   
45  Syrian Arab Republic Humanitarian Response Pla...        0.444822   
16    South Sudan Regional Refugee Response Plan 2024        0.321712   
46  Inter-Agency Emergency Appeal for the Influx f...        0.290070   
14           Ethiopia Humanitarian Response Plan 2024        0.247669   
27  Myanmar Humanitarian Needs and Response Plan 2024        0.220640   
17  Sudan Emergency: Regional Refugee Response Pla...        0.103871   
54              Yemen Humanitarian Response Plan 2024        0.102045   
52       Venezuela Plan de Respuesta Humanitaria 2024        0.100781   
11  Venezuela Regional Refugee and Migrant Respons...        0.095277   

    Severity_Score  percentFunded       In Need  
0         0.976031           52.0  1.956398e+09  
45        0.824822           38.0  6.770727e+08  
16        0.341